# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20 10 — s["b"] берёт значение по ярлыку b (20), а s.iloc[0] — по нулевой позиции (10)
2. Алматы NaN, Астана NaN, Шымкент 3.0,  Pandas складывает Series по ярлыкам. Lля Алматы и Астана нет пары, поэтому получается NaN, а для Шымкент 2 + 1 = 3
3. (2, 2)  фильтр оставляет 2 строки (где x равен 2 и 3) и оба столбца
4. Series DataFrame одни скобки возвращают столбец как Series, двойные скобки со списком как DataFrame
5. [1, 2] 5  фильтр сохраняет старые ярлыки строк 1 и 2, а f.iloc[0, 1] берёт первую по позиции строку и второй столбец (5)

In [16]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [20]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [37]:
# 1
print("Cтрок, столбцов:", df.shape)
print("\nТипы столбцов:\n", df.dtypes)
df.info()
print("\nПропусков в delivery_days:", df.shape[0] - 5303)
print("Разные значения в city:", df["city"].nunique())
print("Уникальных значений в category:", df["category"].nunique())
print("Уникальных значений в status:", df["status"].nunique())
print("Уникальных значений в customer_id:", df["customer_id"].nunique())

Cтрок, столбцов: (5463, 8)

Типы столбцов:
 order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   object 
 2   customer_id    5463 non-null   object 
 3   city           5463 non-null   object 
 4   category       5463 non-null   object 
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 341.6+ KB

Пропусков в delivery_days: 160
Разные значения в city: 14
Уникальных значений в category: 5
Уникальных значений в status: 2
Уникаль

Для менеджера: 1) В столбце delivery_days отсутствуют 160 значений, что может повлиять на анализ сроков доставки
2) Столбец order_date имеет тип object вместо datetime, поэтому перед анализом необходимо преобразовать даты
3) В названиях городов, категорий и статусах могут быть разные варианты написания, что приведёт к неправильной группировке данных и искажению отчёта о выручке


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [27]:
# 2
paid_clean = df.loc[paid & not_test, "amount"]
print("Среднее:", round(paid_clean.mean(), 2))
print("Медиана:", round(paid_clean.median(), 2))
print("25 перцентиль:", round(paid_clean.quantile(0.25), 2))
print("75 перцентиль:", round(paid_clean.quantile(0.75), 2))

Среднее: 14672.39
Медиана: 10300.0
25 перцентиль: 5880.0
75 перцентиль: 18690.0


Для менеджера: Медианный чек составляет 10 300, а средний 14 672,39. Среднее значение может быть завышено изза крупных заказов. Поэтому для отображения типичного чека в отчёте лучше использовать медиану так как она меньше зависит от некорретных данных

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [38]:
# 3
ALM = ["Алматы", "алматы", "АЛМАТЫ", " Алматы", "Almaty"]
cancelled = df["status"] == "отменён"
share_all = cancelled.mean()
share_sh = cancelled[df["city"].isin(SH)].mean()
share_alm = cancelled[df["city"].isin(ALM)].mean()

print(f"Доля отмен по всем заказам: {share_all:.2%}")
print(f"Доля отмен в Шымкенте: {share_sh:.2%}")
print(f"Доля отмен в Алматы: {share_alm:.2%}")
print(f"Шымкент выше Алматы в {share_sh / share_alm:.2f} раза")
print(f"Шымкент выше среднего по всем заказам в {share_sh / share_all:.2f} раза")

Доля отмен по всем заказам: 10.23%
Доля отмен в Шымкенте: 20.40%
Доля отмен в Алматы: 7.86%
Шымкент выше Алматы в 2.59 раза
Шымкент выше среднего по всем заказам в 1.99 раза


Для менеджера: Шымкент выделяется высокой долей отмен это в 2.6 раза выше, чем в Алматы 7.86%, и в 2.0 раза выше среднего по всей базе 10.23%

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [39]:
# 4
base = paid & not_test
sh_mask = df["city"].isin(SH)
alm_mask = df["city"].isin(ALM)
sh_sep = df.loc[base & sep & sh_mask, "amount"].sum()
sh_oct = df.loc[base & octo & sh_mask, "amount"].sum()
sh_diff_pct = (sh_oct - sh_sep) / sh_sep * 100
alm_sep = df.loc[base & sep & alm_mask, "amount"].sum()
alm_oct = df.loc[base & octo & alm_mask, "amount"].sum()
alm_diff_pct = (alm_oct - alm_sep) / alm_sep * 100

print(f"Шымкент: сентябрь = {sh_sep:,.0f}, октябрь = {sh_oct:,.0f}, изменение = {sh_diff_pct:.2f}%")
print(f"Алматы: сентябрь = {alm_sep:,.0f}, октябрь = {alm_oct:,.0f}, изменение = {alm_diff_pct:.2f}%")
print(f"Падение в % в Шымкенте сильнее в {abs(sh_diff_pct) / abs(alm_diff_pct):.2f} раза")

Шымкент: сентябрь = 4,949,800, октябрь = 4,044,010, изменение = -18.30%
Алматы: сентябрь = 15,523,940, октябрь = 14,643,020, изменение = -5.67%
Падение в % в Шымкенте сильнее в 3.22 раза


Для менеджера: В процентах падение в Шымкенте -18.30% сильнее в 3.2 раза, чем в Алматы -5.67%, хотя в деньгах потери сопоставимы 906 тыс и 881 тыс. В первую очередь нужно проверить причины просадки и отмен в Шымкенте, а также убрать дубли

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [40]:
# 5
# 1: оператор & имеет более высокий приоритет, чем ==, поэтому без скобок Python пытается вычислить "Шымкент" & df["status"] и код падает с TypeError
# 2: проверка df["city"] == "Шымкент" учитывает только одно написание и теряет "шымкент" и "Shymkent"

cancel_sh = df[df["city"].isin(SH) & (df["status"] == "отменён")]
print("Отменённых заказов в Шымкенте (все написания):", len(cancel_sh))
print("Только точное 'Шымкент':", len(df[(df["city"] == "Шымкент") & (df["status"] == "отменён")]))

Отменённых заказов в Шымкенте (все написания): 164
Только точное 'Шымкент': 153


Для менеджера: в Шымкенте 164 отменённых заказа. При поиске только по точному слову 'Шымкент' терялись 11 заказов с другим написанием

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [43]:
# 6
vc_customers = df["customer_id"].value_counts()
print("Топ-5 клиентов по количеству записей:\n", vc_customers.head(5))

top_customer = vc_customers.index[0]
top_orders = df.loc[df["customer_id"] == top_customer, ["order_id", "order_date", "city", "amount", "status"]]
print(f"\nВсе заказы клиента {top_customer}:")
print(top_orders)
print("\nВсего строк у клиента:", len(top_orders), "| Уникальных order_id:", top_orders["order_id"].nunique())
print("Частота order_id у клиента:\n", top_orders["order_id"].value_counts().head(3))

Топ-5 клиентов по количеству записей:
 customer_id
C01818    11
C01076     9
C01058     9
C00420     9
C01947     9
Name: count, dtype: int64

Все заказы клиента C01818:
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен

Всего строк у клиента: 11 | Уникальных order_id: 10
Частота order_id у клиента:
 orde

Для менеджера: больше всего записей у клиента C01818, причём заказ 101504 на 8 890 ₸ продублирован дважды. Это значит, что посчитанная без удаления дублей выручка завышена и требует очистки перед финальным отчётом

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [44]:
# 7
q1 = paid_clean.quantile(0.25)
q3 = paid_clean.quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

outliers_iqr = df[paid & not_test & (df["amount"] > upper_bound)]
print(f"Q1 = {q1}, Q3 = {q3}, IQR = {iqr}, верхняя граница = {upper_bound}")
print(f"Заказов выше границы: {len(outliers_iqr)} из {len(paid_clean)} ({len(outliers_iqr) / len(paid_clean):.2%})")
print("\nКатегории выбросов:\n", outliers_iqr["category"].value_counts())

Q1 = 5880.0, Q3 = 18690.0, IQR = 12810.0, верхняя граница = 37905.0
Заказов выше границы: 320 из 4901 (6.53%)

Категории выбросов:
 category
Электроника    308
Одежда          10
Дом              2
Name: count, dtype: int64


Для менеджера: выше границы 37 905 ₸ находится 320 заказов , почти все из категории Электроника. По IQR выбросов больше, чем по z-оценке в ДЗ 4 (87), потому что дорогие чеки раздувают среднее и стандартное отклонение в z-оценке, а квартили к ним устойчивы

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [45]:
# 8
cancel_share_by_city = (df.loc[df["status"] == "отменён", "city"].value_counts() / df["city"].value_counts()).sort_values()
print("Доля отмен по каждому написанию города:\n", cancel_share_by_city)
print("\nКоличество заказов по каждому написанию Шымкента:")
print(df["city"].value_counts().loc[SH])

Доля отмен по каждому написанию города:
 city
АЛМАТЫ       0.032258
шымкент      0.037037
 Алматы      0.045455
Almaty       0.057971
Атырау       0.075342
Алматы       0.079917
Астана       0.084829
Караганда    0.089219
Astana       0.100000
Актобе       0.103275
астана       0.130952
алматы       0.140625
Шымкент      0.215190
Shymkent     0.230769
Name: count, dtype: float64

Количество заказов по каждому написанию Шымкента:
city
Шымкент     711
шымкент      54
Shymkent     39
Name: count, dtype: int64


Для менеджера: Pandas делит значения по ярлыкам городов, но судить по одному написанию нельзя на мелких вариантах вроде 'шымкент' (54 заказа) и 'Shymkent' (39) доля отмен скачет от 3.70% до 23.08%. До анализа названия городов нужно привести к единому виду